# Returns and Indicators

This notebook calculates the **50-day Moving Average (MA50)** for each ETF and adds them to the combined prices dataset.

The output file `prices_with_indicators.csv`.

The simulation notebooks use the MA50 to determine whether to buy or sell on any given day.
The 50-day moving average (MA50) is the average adjusted closing price of an ETF over the past 50 trading days. It is recalculated every day using the most recent 50 days of prices.

This will be used for the timing strategy simulation where:
if the price is > the moving average then investif the price is <= the moving average then move to cash


## 1. Import Libraries and configure paths

In [ ]:
import pandas as pd 
import os   

PROCESSED_DIR = "../data/processed"
INPUT_FILE  = os.path.join(PROCESSED_DIR, "combined_prices.csv")
OUTPUT_FILE = os.path.join(PROCESSED_DIR, "prices_with_indicators.csv")

ma_window = 50 

print(f"Input: {os.path.abspath(INPUT_FILE)}")
print(f"Output: {os.path.abspath(OUTPUT_FILE)}")
print(f"Moving average window: {ma_window} days")

## 2. Load `combined_prices.csv`

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

# Confirm the file loaded correctly
print(f"Loaded combined_prices.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Nulls: {df.isnull().sum().to_dict()}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string(index=False))

## 3. Calculate MA50 for Each ETF

We calculate the 50-day moving average for SPY, QQQ, and VTI using `.rolling(50).mean()`.

How `.rolling(50).mean()` works
For each row, it looks back at the **50 most recent rows** (including the current one) and takes the average price.  
On the next day, the window slides forward by one, the oldest day drops off and the newest day is included.

```
Day 1:   only 1 price available  → NaN  (need 50)
Day 2:   only 2 prices available → NaN  (need 50)
...
Day 49:  only 49 prices          → NaN  (need 50)
Day 50:  50 prices available     → first valid MA50 = average of days 1–50
Day 51:  50 prices available     → MA50 = average of days 2–51
Day 52:  50 prices available     → MA50 = average of days 3–52

Cont.
```

NOTE: The first 49 rows of each MA50 column will be `NaN` 

In [ ]:
df["SPY_MA50"] = df["SPY_Price"].rolling(ma_window).mean()
df["QQQ_MA50"] = df["QQQ_Price"].rolling(ma_window).mean()
df["VTI_MA50"] = df["VTI_Price"].rolling(ma_window).mean()

# Confirm column order matches the data dictionary
# Date | SPY_Price | QQQ_Price | VTI_Price | SPY_MA50 | QQQ_MA50 | VTI_MA50
df = df[["Date", "SPY_Price", "QQQ_Price", "VTI_Price", "SPY_MA50", "QQQ_MA50", "VTI_MA50"]]

# Preview
print(f"MA50 columns added.")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows (MA50 will be NaN):")
print(df.head(3).to_string(index=False))
print(f"\nRow 50 onwards (first valid MA50 values):")
print(df.iloc[48:52].to_string(index=False))
print(f"\nLast 3 rows:")
print(df.tail(3).to_string(index=False))

## 4. Validation

Verify:
1. **Columns:** the 7 required columns in the correct order
2. **MA50 NaN count:** should be exactly 49 NaNs per MA50 column (rows 0–48)
3. **MA50 values are positive** — a moving average must be positive
4. **Row count:** we should have the same number of rows as `combined_prices.csv`


In [ ]:
print("  VALIDATION — prices_with_indicators.csv")

# Columns validation
expected_cols = ["Date", "SPY_Price", "QQQ_Price", "VTI_Price",
                 "SPY_MA50", "QQQ_MA50", "VTI_MA50"]

print(f"\nColumn Check")
print(f"Expected: {expected_cols}")
print(f"Actual:   {list(df.columns)}")
status = "PASS" if list(df.columns) == expected_cols else "FAIL: column mismatch"
print(f"{status}")

# MA50 NaN count validation
# First 49 rows should be NaN 
print(f"\nMA50 NaN Count")
for col in ["SPY_MA50", "QQQ_MA50", "VTI_MA50"]:
    nan_count = df[col].isna().sum()
    status    = "PASS" if nan_count == 49 else f"FAIL: {nan_count} NaNs"
    print(f"{col}: {nan_count} NaNs, {status}")

# MA50 values are positive 
# Drop NaN rows before checking because we only check rows where MA50 has a value
print(f"\nMA50 Positive Value Check")
for col in ["SPY_MA50", "QQQ_MA50", "VTI_MA50"]:
    bad = (df[col].dropna() <= 0).sum()
    status = "PASS" if bad == 0 else f"FAIL: {bad} zero or negative values"
    print(f"{col}: {status}")

# Row count validation
print(f"\nRow Count")
print(f"Total rows: {len(df)}")

## 5.  Save `prices_with_indicators.csv`

Save the final file to `data/processed/`.  

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved prices_with_indicators.csv")
print(f"Path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")